In [1]:
import re
import sys
import json
import numpy as np
from cobra.io import validate_sbml_model
from cobra.io import read_sbml_model
from cobra.io import write_sbml_model


from cobra.flux_analysis import flux_variability_analysis

sys.path.append("modules/")

from utils import correct_ids, trimm_model

In [2]:
# In the original model I manually fixed a GPR
# GENE_ASSOCIATION: Mbar_A0818 or Mbar_A1889 )
original_model_path = "models/M_barkeri_model_iMG746/mba_model.xml"
output_model_path   = "models/M_barkeri_model_iMG746.xml"
final_model_path   = "models/M_barkeri_model_iMG746_trimmed.xml"

print(f"Reading and validating SBML model from {original_model_path}")
mb_model = read_sbml_model(output_model_path)
mb_model, errors = validate_sbml_model(original_model_path)

assert len(errors["SBML_FATAL"]) == 0
assert len(errors["SBML_ERROR"]) == 0
assert len(errors["SBML_SCHEMA_ERROR"]) == 0

replace_dict = {
    "_LPAREN_":"_", 
    "_RPAREN_":"", 
    "_DASH_":"_" , 
    "_unknown_":"_",
    "unknown_": ""
}

print("Correcting model IDs")
for pattern, fix in replace_dict.items():
    print(f"- Replacing \"{pattern}\" by \"{fix}\" in model's reaction/metabolite ids")
    mb_model = correct_ids(mb_model, pattern, fix)
    print(f"- ID correcting pattern {pattern}.")

print(f"Writing fixed model to {output_model_path}")
write_sbml_model(mb_model, output_model_path)

print(f"Re-reading fixed model from {output_model_path}")
mb_model = read_sbml_model(output_model_path)

print("Final check for inconsistent IDs")
total = 0
for pattern, fix in replace_dict.items():
    for r in mb_model.reactions:
        if re.search(r.id, pattern):
            total += 1
    for m in mb_model.metabolites:
        if re.search(m.id, pattern):
            total += 1

assert total == 0
print("==========================================")
print("Success!!!")

tolerance = 1e-9

Reading and validating SBML model from models/M_barkeri_model_iMG746/mba_model.xml


SBML warnings in validation, check error log for details.
COBRA warnings in validation, check error log for details.


Correcting model IDs
- Replacing "_LPAREN_" by "_" in model's reaction/metabolite ids
N of corrected reaction ids 78
N of corrected metabolites ids 82
- ID correcting pattern _LPAREN_.
- Replacing "_RPAREN_" by "" in model's reaction/metabolite ids
N of corrected reaction ids 78
N of corrected metabolites ids 82
- ID correcting pattern _RPAREN_.
- Replacing "_DASH_" by "_" in model's reaction/metabolite ids
N of corrected reaction ids 4
N of corrected metabolites ids 85
- ID correcting pattern _DASH_.
- Replacing "_unknown_" by "_" in model's reaction/metabolite ids
N of corrected reaction ids 2
N of corrected metabolites ids 2
- ID correcting pattern _unknown_.
- Replacing "unknown_" by "" in model's reaction/metabolite ids
N of corrected reaction ids 0
N of corrected metabolites ids 4
- ID correcting pattern unknown_.
Writing fixed model to models/M_barkeri_model_iMG746.xml
Re-reading fixed model from models/M_barkeri_model_iMG746.xml
Final check for inconsistent IDs
Success!!!


In [3]:
H2_uptake_bound = -20
acetate_uptake_bound = -5

mb_model = read_sbml_model(output_model_path)

mb_model.reactions.get_by_id("EX_meoh_e").lower_bound = 0
mb_model.reactions.get_by_id("EX_cys_L_e").lower_bound = 0

mb_model.reactions.get_by_id("EX_h2_e").lower_bound = H2_uptake_bound

print("Fixing bounds to remove meoh from the media and cysteine and meoh enable H2\n")

mb_model.reactions.get_by_id("EX_ac_e").lower_bound = 0
solution_co2 = mb_model.optimize()
summary_co2 = mb_model.summary()
print(f"Growth rate with co2 as carbon source: {solution_co2.objective_value:.5f}")

mb_model.reactions.get_by_id("EX_ac_e").lower_bound = acetate_uptake_bound
solution_acetate = mb_model.optimize()
summary_acetate = mb_model.summary()
print(f"Growth rate with Acetate (uptake:{acetate_uptake_bound}) as carbon source: {solution_acetate.objective_value:.5f}")


Fixing bounds to remove meoh from the media and cysteine and meoh enable H2

Growth rate with co2 as carbon source: 0.01272
Growth rate with Acetate (uptake:-5) as carbon source: 0.03996


In [4]:
trimmed_model = trimm_model(mb_model)

print("Testing the model")

trimmed_model.reactions.get_by_id("EX_ac_e").lower_bound = 0
new_solution_co2 = trimmed_model.optimize()
print(f"Growth rate with CO2 as carbon source:")
print(f"- Original solution: {solution_co2.objective_value:.5f}")
print(f"- Constrained-model solution: {new_solution_co2.objective_value:.5f}")

diff = solution_co2.objective_value - new_solution_co2.objective_value
assert abs(diff) < 1e-9

trimmed_model.reactions.get_by_id("EX_ac_e").lower_bound = acetate_uptake_bound
new_solution_ac = trimmed_model.optimize()
ac_summary = trimmed_model.summary()
print(f"Growth rate with Acetate (uptake:{acetate_uptake_bound}) as carbon source:")
print(f"- Original solution: {solution_acetate.objective_value:.5f}")
print(f"- Constrained-model solution: {new_solution_ac.objective_value:.5f}")

diff = solution_acetate.objective_value - new_solution_ac.objective_value
assert abs(diff) < 1e-9

print("=============================================")
print("Success!")

print(f"- N reactions in original model: {len(mb_model.reactions)}")
print(f"- N reactions in trimmed model:  {len(trimmed_model.reactions)}")
print()
print(f"- N metabolites in original model: {len(mb_model.metabolites)}")
print(f"- N metabolites in trimmed model:  {len(trimmed_model.metabolites)}")
print()
print(f"- N genes in original model: {len(mb_model.genes)}")
print(f"- N genes in trimmed model:  {len(trimmed_model.genes)}")

print(f"Writing Final model to {final_model_path}")
write_sbml_model(trimmed_model, final_model_path)
print("Success!")

Removing blocked 265 reactions
Removing 259 gap metabolites
Removing 204 unconnected genes
Setting bounds
Testing the model
Growth rate with CO2 as carbon source:
- Original solution: 0.01272
- Constrained-model solution: 0.01272


/home/mponce/projects/devel/PhysiCelldFBA/venv/lib/python3.10/site-packages/cobra/core/group.py:147: UserWarning: need to pass in a list
  warn("need to pass in a list")


Growth rate with Acetate (uptake:-5) as carbon source:
- Original solution: 0.03996
- Constrained-model solution: 0.03996
Success!
- N reactions in original model: 816
- N reactions in trimmed model:  551

- N metabolites in original model: 718
- N metabolites in trimmed model:  459

- N genes in original model: 750
- N genes in trimmed model:  546
Writing Final model to models/M_barkeri_model_iMG746_trimmed.xml
Success!


In [5]:
trimmed_model.reactions.get_by_id("EX_ac_e").lower_bound = 0
new_solution_co2 = trimmed_model.optimize()
co2_summary = trimmed_model.summary()

trimmed_model.reactions.get_by_id("EX_ac_e").lower_bound = acetate_uptake_bound
new_solution_ac = trimmed_model.optimize()
ac_summary = trimmed_model.summary()

In [6]:
from cobra.util.array import create_stoichiometric_matrix

N = create_stoichiometric_matrix(trimmed_model)
n = N.shape[1]
rank = np.linalg.matrix_rank(N)
print(n-rank)

109


In [7]:
model = read_sbml_model(final_model_path)
model.reactions.get_by_id("EX_ac_e").lower_bound = 0
solution = model.optimize()
tolerance = 1e-8
support_co2 = set(solution.fluxes.index[solution.fluxes.abs() > tolerance].tolist())

model.reactions.get_by_id("EX_ac_e").lower_bound = acetate_uptake_bound
solution = model.optimize()
tolerance = 1e-8
support_ac = set(solution.fluxes.index[solution.fluxes.abs() > tolerance].tolist())


print(len(support_ac|support_co2))
print(len(support_ac-support_co2))
print(len(support_co2-support_ac))

394
3
0
